In [1]:
import numpy as np
import pandas as pd
import gensim
import os
import re
from nltk import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess
import nltk

In [2]:
# Download required NLTK data
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
print("NLTK data downloaded successfully!")

NLTK data downloaded successfully!


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bibhu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bibhu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bibhu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# Load IMDB Dataset from the same directory
df = pd.read_csv("IMDB Dataset.csv")
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
# Preprocess the reviews - remove HTML tags and clean text
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

# Apply cleaning
df['cleaned_review'] = df['review'].apply(remove_tags)
df['cleaned_review'] = df['cleaned_review'].apply(lambda x: x.lower())

# Check the cleaned data
print(f"Sample cleaned review:\n{df['cleaned_review'][0][:200]}...")
df[['review', 'cleaned_review', 'sentiment']].head()

Sample cleaned review:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was it...


,review,cleaned_review,sentiment
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,a wonderful little production. the filming tec...,positive
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...","petter mattei's ""love in the time of money"" is...",positive


In [5]:
# Tokenize reviews into sentences and then into words
# Use a subset for faster processing (adjust as needed)
reviews_to_process = df['cleaned_review'][:5000]  # Using first 5000 reviews

sentences = []
for review in reviews_to_process:
    # Split review into sentences
    review_sentences = sent_tokenize(review)
    for sentence in review_sentences:
        # Tokenize each sentence into words
        tokens = simple_preprocess(sentence, deacc=True)
        if len(tokens) > 0:
            sentences.append(tokens)

print(f"Total sentences: {len(sentences)}")
print(f"Sample sentence: {sentences[0]}")

Total sentences: 52385
Sample sentence: ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'll', 'be', 'hooked']


In [6]:
# Check number of sentences
print(f"Number of sentences: {len(sentences)}")
print(f"Average words per sentence: {np.mean([len(s) for s in sentences]):.2f}")

Number of sentences: 52385
Average words per sentence: 20.95


In [7]:
# Display first few sentences
sentences[:5]

[['one',
  'of',
  'the',
  'other',
  'reviewers',
  'has',
  'mentioned',
  'that',
  'after',
  'watching',
  'just',
  'oz',
  'episode',
  'you',
  'll',
  'be',
  'hooked'],
 ['they',
  'are',
  'right',
  'as',
  'this',
  'is',
  'exactly',
  'what',
  'happened',
  'with',
  'me',
  'the',
  'first',
  'thing',
  'that',
  'struck',
  'me',
  'about',
  'oz',
  'was',
  'its',
  'brutality',
  'and',
  'unflinching',
  'scenes',
  'of',
  'violence',
  'which',
  'set',
  'in',
  'right',
  'from',
  'the',
  'word',
  'go'],
 ['trust',
  'me',
  'this',
  'is',
  'not',
  'show',
  'for',
  'the',
  'faint',
  'hearted',
  'or',
  'timid'],
 ['this',
  'show',
  'pulls',
  'no',
  'punches',
  'with',
  'regards',
  'to',
  'drugs',
  'sex',
  'or',
  'violence'],
 ['its',
  'is',
  'hardcore',
  'in',
  'the',
  'classic',
  'use',
  'of',
  'the',
  'word',
  'it',
  'is',
  'called',
  'oz',
  'as',
  'that',
  'is',
  'the',
  'nickname',
  'given',
  'to',
  'the',
  'os

In [8]:
# Display first sentence
sentences[0]

['one',
 'of',
 'the',
 'other',
 'reviewers',
 'has',
 'mentioned',
 'that',
 'after',
 'watching',
 'just',
 'oz',
 'episode',
 'you',
 'll',
 'be',
 'hooked']

In [ ]:
# Create Word2Vec model for IMDB reviews
model = gensim.models.Word2Vec(
    vector_size=100,  # Dimensionality of word vectors
    window=5,         # Context window size
    min_count=5,      # Ignore words with frequency less than 5
    workers=4,        # Number of threads
    sg=0              # 0 for CBOW, 1 for Skip-gram 
)

print("Word2Vec model initialized")

Word2Vec model initialized


In [10]:
# Build vocabulary from sentences
model.build_vocab(sentences)
print(f"Vocabulary size: {len(model.wv)}")

Vocabulary size: 12185


In [11]:
# Train the Word2Vec model
model.train(sentences, total_examples=model.corpus_count, epochs=10)
print("Model training completed!")

Model training completed!


In [12]:
# Find words most similar to 'good'
model.wv.most_similar('good', topn=10)

[('bad', 0.7069160342216492),
 ('great', 0.6910209059715271),
 ('decent', 0.6590898633003235),
 ('cool', 0.6428926587104797),
 ('nice', 0.6295045018196106),
 ('poor', 0.6165058612823486),
 ('funny', 0.5836848616600037),
 ('fine', 0.5603439807891846),
 ('terrible', 0.5447007417678833),
 ('sure', 0.5263873934745789)]

In [13]:
# Calculate similarity between movie-related words
similarity = model.wv.similarity('good', 'bad')
print(f"Similarity between 'good' and 'bad': {similarity:.4f}")

# Try other word pairs
try:
    sim2 = model.wv.similarity('movie', 'film')
    print(f"Similarity between 'movie' and 'film': {sim2:.4f}")
except KeyError as e:
    print(f"Word not in vocabulary: {e}")

Similarity between 'good' and 'bad': 0.7069
Similarity between 'movie' and 'film': 0.9126


In [14]:
# Check word vector shape
try:
    print(f"Vector shape for 'movie': {model.wv['movie'].shape}")
    print(f"Word vector: {model.wv['movie'][:10]}...")  # Show first 10 dimensions
except KeyError:
    print("Word 'movie' not in vocabulary. Trying 'film'...")
    print(f"Vector shape for 'film': {model.wv['film'].shape}")

Vector shape for 'movie': (100,)
Word vector: [-0.07986576  1.8500322   0.52033925 -0.20644203 -0.91816276 -0.35929468
  0.04277129  3.4282222   0.29061848  1.1581799 ]...


In [15]:
vec = model.wv.get_normed_vectors()

In [16]:
vec

array([[-0.0922096 ,  0.11068641,  0.07339866, ..., -0.01879523,
         0.1209376 ,  0.05164171],
       [-0.08832585,  0.13083535, -0.03744165, ..., -0.11484639,
        -0.09519952,  0.07415597],
       [ 0.07565916,  0.10297157, -0.02722836, ..., -0.06722557,
         0.00509234,  0.07745052],
       ...,
       [-0.084721  ,  0.07229789,  0.02354535, ..., -0.10784259,
         0.07667268, -0.05729543],
       [ 0.00334274,  0.1564472 , -0.03881286, ..., -0.18401365,
         0.02406017,  0.01718173],
       [ 0.01063996, -0.01195752,  0.00901699, ..., -0.18811378,
         0.10331721, -0.00900063]], shape=(12185, 100), dtype=float32)

In [17]:
model.wv.get_normed_vectors().shape

(12185, 100)

In [18]:

y = model.wv.index_to_key

In [19]:
len(y)

12185

In [20]:
y

['the',
 'and',
 'of',
 'to',
 'is',
 'it',
 'in',
 'this',
 'that',
 'was',
 'as',
 'movie',
 'with',
 'for',
 'but',
 'film',
 'you',
 'on',
 'not',
 'are',
 'his',
 'he',
 'have',
 'be',
 'one',
 'at',
 'all',
 'they',
 'by',
 'an',
 'who',
 'so',
 'from',
 'like',
 'there',
 'or',
 'just',
 'about',
 'her',
 'if',
 'out',
 'what',
 'has',
 'some',
 'good',
 'can',
 'when',
 'very',
 'more',
 'she',
 'up',
 'no',
 'would',
 'even',
 'time',
 'which',
 'my',
 'story',
 'see',
 'their',
 'really',
 'only',
 'had',
 'well',
 'were',
 'we',
 'me',
 'much',
 'been',
 'than',
 'get',
 'because',
 'bad',
 'into',
 'do',
 'will',
 'other',
 'first',
 'people',
 'also',
 'how',
 'great',
 'don',
 'him',
 'most',
 'way',
 'made',
 'make',
 'them',
 'then',
 'movies',
 'too',
 'any',
 'its',
 'could',
 'after',
 'watch',
 'characters',
 'being',
 'think',
 'character',
 'never',
 'films',
 'plot',
 'little',
 'seen',
 'many',
 'two',
 'life',
 'where',
 'acting',
 'best',
 'know',
 'love',
 'y

In [21]:

from sklearn.decomposition import PCA

In [22]:

pca = PCA(n_components=3)

In [23]:

X = pca.fit_transform(model.wv.get_normed_vectors())

In [24]:
X

array([[ 0.55544955,  0.11338023, -0.10575926],
       [ 0.31930646,  0.02582681,  0.09065   ],
       [ 0.46986908,  0.0299892 , -0.11542362],
       ...,
       [-0.05154198, -0.363291  ,  0.07131994],
       [-0.08177257, -0.10314491, -0.04901101],
       [-0.11891174,  0.01305858, -0.14825845]],
      shape=(12185, 3), dtype=float32)

In [25]:
X.shape

(12185, 3)

In [26]:
# Visualize word embeddings in 3D space
import plotly.express as px

# Select a subset of words for visualization (sentiment-related words)
words_to_plot = ['good', 'bad', 'great', 'terrible', 'excellent', 'awful', 
                 'amazing', 'horrible', 'best', 'worst', 'love', 'hate',
                 'beautiful', 'ugly', 'perfect', 'poor', 'wonderful', 'disappointing']

# Filter words that exist in vocabulary
available_words = [w for w in words_to_plot if w in model.wv]
print(f"Visualizing {len(available_words)} words: {available_words}")

# Get word vectors for available words
word_indices = [model.wv.key_to_index[w] for w in available_words]
vectors = model.wv.get_normed_vectors()[word_indices]

# Reduce to 3D using PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=3)
vectors_3d = pca.fit_transform(vectors)

# Create 3D scatter plot
fig = px.scatter_3d(
    x=vectors_3d[:, 0], 
    y=vectors_3d[:, 1], 
    z=vectors_3d[:, 2],
    text=available_words,
    title="Word2Vec Embeddings - IMDB Sentiment Words"
)
fig.update_traces(textposition='top center', marker=dict(size=8))
fig.show()

Visualizing 18 words: ['good', 'bad', 'great', 'terrible', 'excellent', 'awful', 'amazing', 'horrible', 'best', 'worst', 'love', 'hate', 'beautiful', 'ugly', 'perfect', 'poor', 'wonderful', 'disappointing']
